In [ ]:

from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
from IPython.display import display

sns.set_theme(style='whitegrid')
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', None)

BASE_DIR = Path.cwd()
PROJECT_ROOT = BASE_DIR.parent if BASE_DIR.name == 'noteboods' else BASE_DIR
DATA_DIR = PROJECT_ROOT / 'data'

plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11


In [ ]:

df = pd.read_csv(DATA_DIR / 'df.csv')
df_map = pd.read_csv(DATA_DIR / 'df_map.csv')

with open(DATA_DIR / 'chicago_community_areas.geojson', 'r') as f:
    community_geojson = json.load(f)

raw_cleaning_summary = {
    'raw_rows_after_concat': 2491875,
    'final_modeling_rows': 2491668,
    'final_mapping_rows': 2454715,
    'location_description_missing_raw': 13168,
    'district_missing_raw': 1,
    'community_area_missing_raw': 206,
    'latitude_missing_raw': 36957,
    'longitude_missing_raw': 36957,
}


In [ ]:

def fix_dtypes(data):
    data = data.copy()
    data['Date'] = pd.to_datetime(data['Date'], errors='coerce')

    int_cols = ['ID', 'District', 'Community Area', 'Year', 'Month', 'Hour', 'DayOfWeek']
    for col in int_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors='coerce').astype('Int32')

    float_cols = ['Latitude', 'Longitude']
    for col in float_cols:
        if col in data.columns:
            data[col] = pd.to_numeric(data[col], errors='coerce').astype('float32')

    bool_cols = ['Arrest', 'Domestic']
    for col in bool_cols:
        if col in data.columns:
            data[col] = data[col].astype('boolean')

    cat_cols = ['Primary Type', 'Description', 'Location Description']
    for col in cat_cols:
        if col in data.columns:
            data[col] = data[col].astype('category')

    return data


df = fix_dtypes(df)
df_map = fix_dtypes(df_map)



## 1. Dataset Overview

This project uses the City of Chicago's public `Crimes - 2001 to Present` records, restricted to **2016-2025 full-year data**. The incomplete 2026 records were excluded during data cleaning, so all analysis and modeling in this notebook are based on complete yearly observations only.

The EDA relies on the cleaned outputs from `Data_cleaning.ipynb`, which defines the study sample used for both modeling and mapping.


In [ ]:

overview_summary = pd.DataFrame([
    ['Raw combined sample from 2016-2025 CSV files', raw_cleaning_summary['raw_rows_after_concat']],
    ['Final analysis / modeling sample (df)', raw_cleaning_summary['final_modeling_rows']],
    ['Final map-ready sample with valid coordinates (df_map)', raw_cleaning_summary['final_mapping_rows']],
    ['Number of complete years used', 10],
], columns=['Metric', 'Value'])

overview_summary



The raw crime files were cleaned using the following rules:

1. Keep the core fields required for arrest prediction: `ID`, `Date`, `Primary Type`, `Description`, `Location Description`, `Arrest`, `Domestic`, `District`, `Community Area`, `Year`, `Latitude`, and `Longitude`.
2. Fill missing `Location Description` values with `Unknown`.
3. Drop observations missing `District` or `Community Area`, because these are important spatial predictors for both EDA and downstream modeling.
4. Create `df_map` by additionally requiring valid `Latitude` and `Longitude` for spatial visualization.
5. Derive `Month`, `Hour`, and `DayOfWeek` from `Date`.
6. Standardize variable types for dates, integers, booleans, floats, and categorical fields.


In [ ]:

raw_missing_summary = pd.DataFrame({
    'Missing Count (Raw Combined Sample)': {
        'Location Description': raw_cleaning_summary['location_description_missing_raw'],
        'District': raw_cleaning_summary['district_missing_raw'],
        'Community Area': raw_cleaning_summary['community_area_missing_raw'],
        'Latitude': raw_cleaning_summary['latitude_missing_raw'],
        'Longitude': raw_cleaning_summary['longitude_missing_raw'],
    }
})
raw_missing_summary['Missing Rate % (Raw Combined Sample)'] = (
    raw_missing_summary['Missing Count (Raw Combined Sample)']
    / raw_cleaning_summary['raw_rows_after_concat'] * 100
).round(3)

clean_missing_summary = pd.DataFrame({
    'Missing Count in df': df.isna().sum(),
    'Missing Count in df_map': df_map.isna().sum(),
})
clean_missing_summary = clean_missing_summary.loc[
    ['Location Description', 'District', 'Community Area', 'Latitude', 'Longitude']
]
clean_missing_summary['Missing Rate in df %'] = (clean_missing_summary['Missing Count in df'] / len(df) * 100).round(3)
clean_missing_summary['Missing Rate in df_map %'] = (clean_missing_summary['Missing Count in df_map'] / len(df_map) * 100).round(3)

raw_missing_summary, clean_missing_summary


In [ ]:

quality_summary = pd.DataFrame([
    ['Duplicate IDs in df', int(df['ID'].duplicated().sum())],
    ['Duplicate IDs in df_map', int(df_map['ID'].duplicated().sum())],
    ['Unique crime types', int(df['Primary Type'].nunique())],
    ['Unique location descriptions', int(df['Location Description'].nunique())],
    ['Unique police districts', int(df['District'].nunique())],
    ['Unique community areas', int(df['Community Area'].nunique())],
], columns=['Metric', 'Value'])

quality_summary



The main data quality issues are limited and manageable. In the raw combined sample, the largest missingness occurs in `Location Description` and in latitude/longitude fields. After cleaning, the modeling dataset `df` has complete values for all predictors except coordinates, while `df_map` is fully complete for mapping variables. In addition, `ID` is unique in both cleaned datasets, so duplication is not a concern.

A measurement caveat remains for the timestamp variable. As shown later in the hourly analysis, crime records are disproportionately concentrated at exact hour marks, especially midnight, which suggests that some incident times are rounded, defaulted, or recorded imprecisely. For that reason, time-of-day features are still useful, but their interpretation should remain cautious.



## 2. Overall Crime Patterns

This section provides a concise background view of how crime incidents are distributed across time and crime categories. The goal is not just to describe crime counts, but to establish the context in which arrest prediction will later be modeled.


In [ ]:

year_counts = df.groupby('Year').size().reset_index(name='Crime Count')
monthly_by_year = df.groupby(['Year', 'Month']).size().reset_index(name='Crime Count')
monthly_avg = monthly_by_year.groupby('Month')['Crime Count'].mean().reset_index(name='Average Crime Count')
hour_counts = df.groupby('Hour').size().reset_index(name='Crime Count')

day_map = {0: 'Mon', 1: 'Tue', 2: 'Wed', 3: 'Thu', 4: 'Fri', 5: 'Sat', 6: 'Sun'}
day_order = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
df['DayName'] = df['DayOfWeek'].map(day_map)

day_counts = (
    df.groupby('DayName')
      .size()
      .reindex(day_order)
      .reset_index(name='Crime Count')
)


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].plot(year_counts['Year'], year_counts['Crime Count'], marker='o')
axes[0, 0].set_title('Crime Count by Year')
axes[0, 0].set_xlabel('Year')
axes[0, 0].set_ylabel('Crime Count')
axes[0, 0].set_xticks(year_counts['Year'])

axes[0, 1].plot(monthly_avg['Month'], monthly_avg['Average Crime Count'], marker='o', color='darkorange')
axes[0, 1].set_title('Average Crime Count by Month')
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Average Crime Count')
axes[0, 1].set_xticks(range(1, 13))

axes[1, 0].plot(hour_counts['Hour'], hour_counts['Crime Count'], marker='o', color='seagreen')
axes[1, 0].set_title('Crime Count by Hour')
axes[1, 0].set_xlabel('Hour')
axes[1, 0].set_ylabel('Crime Count')
axes[1, 0].set_xticks(range(0, 24, 2))

axes[1, 1].plot(day_counts['DayName'], day_counts['Crime Count'], marker='o', color='slateblue')
axes[1, 1].set_title('Crime Count by Day of Week')
axes[1, 1].set_xlabel('Day of Week')
axes[1, 1].set_ylabel('Crime Count')

plt.tight_layout()
plt.show()


In [ ]:

top15_types = df['Primary Type'].value_counts().head(15)

plt.figure(figsize=(9, 6))
top15_types.sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 15 Crime Types by Frequency')
plt.xlabel('Crime Count')
plt.ylabel('Primary Type')
plt.tight_layout()
plt.show()


In [ ]:

minute_summary = (
    df['Date'].dt.minute.value_counts(normalize=True)
      .sort_index()
      .mul(100)
      .round(2)
      .rename('Percent of records')
)
minute_summary.head(10), minute_summary.loc[[0, 15, 30, 45]].dropna()



Crime volume is fairly stable before the pandemic period, drops sharply in 2020-2021, and then rebounds. Monthly patterns show clear seasonality, with lower crime levels in winter and higher levels in summer. Crime also tends to rise on weekends and during later daytime and evening hours.

At the same time, the time field should not be interpreted too literally. The unusually large share of records at exact minute values, especially minute `00`, suggests recording imprecision. For prediction, this means temporal variables remain useful, but their signal may partly reflect reporting processes in addition to real behavioral timing.



## 3. Arrest Patterns

This section shifts the focus from overall crime volume to the target variable of interest: whether an incident leads to an arrest. The central EDA question is which temporal, categorical, spatial, and situational factors are associated with differences in arrest probability.



### 3.1 Overall Arrest Rate


In [ ]:

overall_arrest_rate = float(df['Arrest'].mean())
arrest_summary = pd.DataFrame([
    ['Total incidents', len(df)],
    ['Arrested incidents', int(df['Arrest'].sum())],
    ['Overall arrest rate', round(overall_arrest_rate, 4)],
    ['Overall arrest rate (%)', round(overall_arrest_rate * 100, 2)],
], columns=['Metric', 'Value'])

arrest_summary



The overall arrest rate is about **16.5%**, which means this is a clearly imbalanced classification problem: non-arrest incidents dominate the data. That matters for modeling, because accuracy alone would be misleading, and class-aware evaluation metrics will be necessary in later stages.



### 3.2 Arrest Patterns by Crime Type


In [ ]:

crime_type_summary = (
    df.groupby('Primary Type')
      .agg(
          Total=('Arrest', 'size'),
          Arrested=('Arrest', 'sum')
      )
      .reset_index()
)
crime_type_summary['Arrest Rate'] = crime_type_summary['Arrested'] / crime_type_summary['Total']
crime_type_summary = crime_type_summary.sort_values('Total', ascending=False)

crime_type_top_count = crime_type_summary.head(15).copy()
crime_type_top_rate = (
    crime_type_summary[crime_type_summary['Total'] >= 10000]
      .sort_values('Arrest Rate', ascending=False)
      .head(12)
      .sort_values('Arrest Rate')
)
crime_type_summary.head()


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(
    data=crime_type_top_count.sort_values('Total'),
    x='Total',
    y='Primary Type',
    color='steelblue',
    ax=axes[0]
)
axes[0].set_title('Top Crime Types by Frequency')
axes[0].set_xlabel('Crime Count')
axes[0].set_ylabel('Primary Type')

sns.barplot(
    data=crime_type_top_rate,
    x='Arrest Rate',
    y='Primary Type',
    color='darkorange',
    ax=axes[1]
)
axes[1].set_title('Highest Arrest Rates by Crime Type (>=10,000 incidents)')
axes[1].set_xlabel('Arrest Rate')
axes[1].set_ylabel('Primary Type')

plt.tight_layout()
plt.show()


In [ ]:

plt.figure(figsize=(8, 6))
plt.scatter(crime_type_summary['Total'], crime_type_summary['Arrest Rate'], alpha=0.8)
plt.xscale('log')
plt.xlabel('Crime Count (log scale)')
plt.ylabel('Arrest Rate')
plt.title('Crime Frequency vs Arrest Rate by Crime Type')
plt.tight_layout()
plt.show()

crime_type_summary[['Total', 'Arrest Rate']].corr()



Arrest rates differ sharply across crime types. Some common offenses, such as theft, deceptive practice, and motor vehicle theft, have relatively low arrest rates, while categories such as narcotics, weapons violations, and criminal trespass have much higher arrest probabilities.

This is one of the strongest signals in the notebook and strongly supports using offense-based features in the prediction task. The negative relationship between frequency and arrest rate also suggests that high-volume crime categories are not necessarily the easiest ones to clear through arrest.



### 3.3 Arrest Patterns by Time


In [ ]:

hour_summary = (
    df.groupby('Hour')
      .agg(Total=('Arrest', 'size'), Arrested=('Arrest', 'sum'))
      .reset_index()
)
hour_summary['Arrest Rate'] = hour_summary['Arrested'] / hour_summary['Total']

day_summary = (
    df.groupby('DayName')
      .agg(Total=('Arrest', 'size'), Arrested=('Arrest', 'sum'))
      .reindex(day_order)
      .reset_index()
)
day_summary['Arrest Rate'] = day_summary['Arrested'] / day_summary['Total']

monthly_summary = (
    df.groupby('Month')
      .agg(Total=('Arrest', 'size'), Arrested=('Arrest', 'sum'))
      .reset_index()
      .sort_values('Month')
)
monthly_summary['Arrest Rate'] = monthly_summary['Arrested'] / monthly_summary['Total']


In [ ]:

fig, axes = plt.subplots(3, 1, figsize=(12, 15))

for ax, summary, xcol, title in [
    (axes[0], hour_summary, 'Hour', 'Crime Count and Arrest Rate by Hour'),
    (axes[1], day_summary, 'DayName', 'Crime Count and Arrest Rate by Day of Week'),
    (axes[2], monthly_summary, 'Month', 'Crime Count and Arrest Rate by Month'),
]:
    ax.bar(summary[xcol], summary['Total'], color='lightgray', label='Total Crimes')
    ax.bar(summary[xcol], summary['Arrested'], color='orange', label='Arrested')
    ax.set_title(title)
    ax.set_ylabel('Crime Count')
    ax2 = ax.twinx()
    ax2.plot(summary[xcol], summary['Arrest Rate'], color='crimson', marker='o', linewidth=2, label='Arrest Rate')
    ax2.set_ylabel('Arrest Rate')
    if xcol == 'Hour':
        ax.set_xticks(range(0, 24, 2))
    if xcol == 'Month':
        ax.set_xticks(range(1, 13))

plt.tight_layout()
plt.show()


In [ ]:

crime_pivot = df.pivot_table(index='DayName', columns='Hour', values='ID', aggfunc='count').reindex(day_order).astype(float)
arrest_pivot = df.pivot_table(index='DayName', columns='Hour', values='Arrest', aggfunc='mean').reindex(day_order).astype(float)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.heatmap(crime_pivot, cmap='Reds', linewidths=0.3, ax=axes[0])
axes[0].set_title('Crime Count Heatmap (Hour x Day)')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('Day of Week')

sns.heatmap(arrest_pivot, cmap='Blues', linewidths=0.3, ax=axes[1])
axes[1].set_title('Arrest Rate Heatmap (Hour x Day)')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Day of Week')

plt.tight_layout()
plt.show()



Temporal variables matter, but they matter in different ways. Crime counts peak in summer and on weekends, while arrest rates do not perfectly follow the same pattern. In fact, arrest rates are relatively higher in the first months of the year and lower in mid-to-late summer and fall.

This distinction is important for modeling: a variable that predicts crime volume does not necessarily predict arrest probability in the same direction. The hour-by-day heatmaps also suggest interaction effects, meaning that the predictive contribution of time may depend on when during the week an incident occurs.



### 3.4 Interaction Effects Relevant for Modeling


In [ ]:

top_types_for_interaction = crime_type_summary.head(10)['Primary Type']

type_year_pivot = (
    df[df['Primary Type'].isin(top_types_for_interaction)]
      .pivot_table(index='Primary Type', columns='Year', values='Arrest', aggfunc='mean')
      .reindex(top_types_for_interaction)
      .astype(float)
)

plt.figure(figsize=(12, 6))
sns.heatmap(type_year_pivot, cmap='YlGnBu', annot=False, linewidths=0.3)
plt.title('Arrest Rate Heatmap: Primary Type x Year (Top 10 crime types)')
plt.xlabel('Year')
plt.ylabel('Primary Type')
plt.tight_layout()
plt.show()



The relationship between crime type and arrest is not fully stable over time. Some offense categories show much larger shifts in arrest rate across years than others, which reinforces the value of out-of-time validation and suggests that simple static averages may hide temporal heterogeneity.


In [ ]:

top_types_for_loc = crime_type_summary.head(8)['Primary Type']
top_locations_for_interaction = location_order = df['Location Description'].value_counts().head(10).index

type_location_pivot = (
    df[
        df['Primary Type'].isin(top_types_for_loc)
        & df['Location Description'].isin(top_locations_for_interaction)
    ]
    .pivot_table(index='Primary Type', columns='Location Description', values='Arrest', aggfunc='mean')
    .reindex(top_types_for_loc)
    .astype(float)
)

plt.figure(figsize=(14, 6))
sns.heatmap(type_location_pivot, cmap='magma', linewidths=0.3)
plt.title('Arrest Rate Heatmap: Primary Type x Location Description')
plt.xlabel('Location Description')
plt.ylabel('Primary Type')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()



The offense-setting interaction adds another layer of signal. Some crime types have very different arrest probabilities depending on where they occur, which means that using only a main-effect representation of crime type or location would miss part of the structure in the data. This is exactly the kind of heterogeneity that more flexible models can capture better than a purely descriptive summary.



### 3.5 Arrest Patterns by Space and Setting



#### 3.5.1 Location Description


In [ ]:

location_summary = (
    df.groupby('Location Description')
      .agg(Total=('Arrest', 'size'), ArrestRate=('Arrest', 'mean'))
      .reset_index()
      .sort_values('Total', ascending=False)
)

location_top12 = location_summary.head(12).copy()
location_rate_top = (
    location_summary[location_summary['Total'] >= 10000]
      .sort_values('ArrestRate', ascending=False)
      .head(10)
      .sort_values('ArrestRate')
)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(
    data=location_top12.sort_values('Total'),
    x='Total',
    y='Location Description',
    color='steelblue',
    ax=axes[0]
)
axes[0].set_title('Top Location Types by Crime Count')
axes[0].set_xlabel('Crime Count')
axes[0].set_ylabel('Location Description')

sns.barplot(
    data=location_rate_top,
    x='ArrestRate',
    y='Location Description',
    color='darkorange',
    ax=axes[1]
)
axes[1].set_title('Highest Arrest Rates by Location Type (>=10,000 incidents)')
axes[1].set_xlabel('Arrest Rate')
axes[1].set_ylabel('Location Description')

plt.tight_layout()
plt.show()



Location description captures situational context that broad geography cannot fully explain. Public-facing or highly visible settings such as sidewalks, grocery stores, department stores, alleys, and gas stations tend to have much higher arrest rates than residential locations such as apartments, residences, or garages.

That pattern makes `Location Description` a strong candidate feature for prediction because it likely reflects surveillance, witness availability, and the probability that officers or third parties observe the event in real time.



#### 3.5.2 Community Area Maps


In [ ]:

community_lookup = pd.DataFrame([
    {
        'Community Area': int(feature['properties']['area_numbe']),
        'Community Name': feature['properties']['community'].title()
    }
    for feature in community_geojson['features']
])

community_summary = (
    df.groupby('Community Area')
      .agg(Total=('Arrest', 'size'), Arrested=('Arrest', 'sum'))
      .reset_index()
)
community_summary['ArrestRate'] = community_summary['Arrested'] / community_summary['Total']
community_summary = community_summary.merge(community_lookup, on='Community Area', how='left')
community_summary.head()


In [ ]:

community_stats = community_summary.set_index('Community Area')[['Community Name', 'Total', 'ArrestRate']].to_dict('index')

enriched_geojson = json.loads(json.dumps(community_geojson))
for feature in enriched_geojson['features']:
    area = int(feature['properties']['area_numbe'])
    stats = community_stats.get(area, {})
    feature['properties']['community_name'] = stats.get('Community Name', feature['properties'].get('community', 'Unknown').title())
    feature['properties']['crime_count'] = int(stats.get('Total', 0))
    feature['properties']['arrest_rate'] = float(stats.get('ArrestRate', 0.0))


In [ ]:

community_arrest_map = folium.Map(location=[41.85, -87.68], zoom_start=10, tiles='CartoDB positron')

folium.Choropleth(
    geo_data=enriched_geojson,
    data=community_summary,
    columns=['Community Area', 'ArrestRate'],
    key_on='feature.properties.area_numbe',
    fill_color='YlOrRd',
    fill_opacity=0.75,
    line_opacity=0.25,
    legend_name='Arrest Rate by Community Area'
).add_to(community_arrest_map)

folium.GeoJson(
    enriched_geojson,
    style_function=lambda x: {'fillColor': 'transparent', 'color': '#555555', 'weight': 0.5},
    tooltip=folium.GeoJsonTooltip(
        fields=['community_name', 'crime_count', 'arrest_rate'],
        aliases=['Community Area', 'Crime Count', 'Arrest Rate'],
        localize=True,
        sticky=False
    )
).add_to(community_arrest_map)

community_arrest_map


In [ ]:

community_count_map = folium.Map(location=[41.85, -87.68], zoom_start=10, tiles='CartoDB positron')

folium.Choropleth(
    geo_data=enriched_geojson,
    data=community_summary,
    columns=['Community Area', 'Total'],
    key_on='feature.properties.area_numbe',
    fill_color='PuBu',
    fill_opacity=0.75,
    line_opacity=0.25,
    legend_name='Crime Count by Community Area'
).add_to(community_count_map)

folium.GeoJson(
    enriched_geojson,
    style_function=lambda x: {'fillColor': 'transparent', 'color': '#555555', 'weight': 0.5},
    tooltip=folium.GeoJsonTooltip(
        fields=['community_name', 'crime_count', 'arrest_rate'],
        aliases=['Community Area', 'Crime Count', 'Arrest Rate'],
        localize=True,
        sticky=False
    )
).add_to(community_count_map)

community_count_map



The community-area maps show that spatial variation in arrest outcomes is not the same thing as spatial variation in crime volume. Some communities with high incident counts do not have especially high arrest rates, while other communities stand out precisely because arrests occur more frequently conditional on reported crime.

This is a strong argument for geography-aware modeling. `Community Area` is not just a location label; it likely captures neighborhood composition, land use, enforcement patterns, and local offense mix that affect arrest likelihood.



#### 3.5.3 Point-Based Crime Hotspots


In [ ]:

heatmap_sample = df_map.sample(n=min(120000, len(df_map)), random_state=42)
arrest_heatmap_sample = df_map[df_map['Arrest']].sample(
    n=min(60000, int(df_map['Arrest'].sum())),
    random_state=42
)

center_lat = float(df_map['Latitude'].mean())
center_lon = float(df_map['Longitude'].mean())


In [ ]:

crime_density_map = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles='CartoDB positron')
HeatMap(
    heatmap_sample[['Latitude', 'Longitude']].dropna().values.tolist(),
    radius=8,
    blur=6,
    min_opacity=0.35
).add_to(crime_density_map)
crime_density_map


In [ ]:

arrest_density_map = folium.Map(location=[center_lat, center_lon], zoom_start=10, tiles='CartoDB positron')
HeatMap(
    arrest_heatmap_sample[['Latitude', 'Longitude']].dropna().values.tolist(),
    radius=8,
    blur=6,
    min_opacity=0.35
).add_to(arrest_density_map)
arrest_density_map



The point-based hotspot maps complement the community-area choropleths. They show where incidents cluster geographically at a finer level than administrative boundaries, while the arrest-only hotspot map highlights where police-cleared incidents are especially concentrated.

Taken together, the two map styles suggest that spatial signal exists at multiple levels: broad neighborhood context and localized hotspot structure. That strengthens the case for including both area-level and coordinate-derived spatial features in later modeling if needed.



## 4. Arrest Patterns Over Time

The project proposal emphasizes **out-of-time validation**, so this section checks whether arrest outcomes are temporally stable across years. If the answer is no, then random train-test splits would be overly optimistic for predictive evaluation.


In [ ]:

year_summary = (
    df.groupby('Year')
      .agg(Total=('Arrest', 'size'), Arrested=('Arrest', 'sum'))
      .reset_index()
      .sort_values('Year')
)
year_summary['ArrestRate'] = year_summary['Arrested'] / year_summary['Total']
year_summary


In [ ]:

fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(year_summary['Year'], year_summary['Total'], color='lightgray', label='Total Crimes')
ax1.set_xlabel('Year')
ax1.set_ylabel('Crime Count')
ax1.set_xticks(year_summary['Year'])

ax2 = ax1.twinx()
ax2.plot(year_summary['Year'], year_summary['ArrestRate'], color='crimson', marker='o', linewidth=2, label='Arrest Rate')
ax2.set_ylabel('Arrest Rate')

plt.title('Crime Count and Arrest Rate by Year')
plt.tight_layout()
plt.show()



Arrest rates are clearly not stable across time. They are relatively high from 2016 through 2019, then fall sharply beginning in 2020 and recover only partially in 2024 and 2025. This means later-year arrest outcomes are generated under a different regime than the pre-2020 period.

That result directly supports the proposal's out-of-time validation strategy. A random split would mix together records from materially different periods and likely overstate model performance, while year-based splits better reflect the intended forecasting setting.



## 5. EDA Summary for Modeling

The exploratory analysis shows that arrest outcomes vary systematically across several dimensions rather than being driven by a single factor.

**Stronger apparent signals**
- `Primary Type`
- `Location Description`
- `Community Area` and broader spatial structure
- `Year`

**Moderate but still useful signals**
- `Hour`
- `Month`
- `DayOfWeek`

There is also meaningful overlap across some variables. `District` and `Community Area` both encode geography at different levels. `Primary Type` and `Description` are closely related, with `Description` providing more granular detail. All temporal predictors ultimately come from `Date`, so they are different views of the same source field rather than fully independent signals.

Overall, the EDA supports the feature design in the project approval form: temporal, categorical, spatial, and situational variables all appear relevant for arrest prediction. It also provides two clear modeling implications: the target is imbalanced, and arrest patterns shift over time strongly enough that year-based evaluation is necessary.
